In [1]:
# ============================================================
# FINAL INTELLIGENCE LAYER SIGN-OFF
# PART 1 — INTEGRATED MODEL + CORRECTNESS VALIDATION
# ============================================================

import os
import time
import json
import uuid
import hashlib
import warnings
import threading
import concurrent.futures

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

np.random.seed(42)

# ============================================================
# 1. LOAD DATASET
# ============================================================

students = pd.read_csv("../datasets/students.csv")
jobs = pd.read_csv("../datasets/jobs.csv")
matches = pd.read_csv("../datasets/matches.csv")

print("="*110)
print("DATASET LOADED")
print("="*110)

print("Students:", students.shape)
print("Jobs:", jobs.shape)
print("Matches:", matches.shape)

# ============================================================
# 2. MERGE DATA
# ============================================================

data = matches.merge(
    students,
    on="student_id",
    how="inner"
)

data = data.merge(
    jobs,
    on="job_id",
    how="inner"
)

print("\nMerged Dataset:", data.shape)

# ============================================================
# 3. ROBUST FEATURE ENGINEERING
# ============================================================

required_numeric_features = [

    "skill_overlap_count",
    "skill_overlap_ratio",
    "experience_gap",
    "internship_months"

]

for column in required_numeric_features:

    if column not in data.columns:

        data[column] = 0

    data[column] = pd.to_numeric(

        data[column],

        errors="coerce"

    ).fillna(0)

# ------------------------------------------------------------
# LOCATION MATCH
# ------------------------------------------------------------

if (

    "location_x" in data.columns

    and

    "location_y" in data.columns

):

    data["location_match"] = (

        data["location_x"]

        .astype(str)

        .str.lower()

        ==

        data["location_y"]

        .astype(str)

        .str.lower()

    ).astype(int)

else:

    data["location_match"] = 0

# ------------------------------------------------------------
# ROLE MATCH
# ------------------------------------------------------------

if (

    "preferred_role" in data.columns

    and

    "job_title" in data.columns

):

    data["role_match"] = (

        data["preferred_role"]

        .astype(str)

        .str.lower()

        ==

        data["job_title"]

        .astype(str)

        .str.lower()

    ).astype(int)

else:

    data["role_match"] = 0

# ------------------------------------------------------------
# EXPERIENCE SCORE
# ------------------------------------------------------------

experience_max = max(

    data["experience_gap"].max(),

    1

)

data["experience_score"] = (

    1

    -

    data["experience_gap"].clip(lower=0)

    /

    experience_max

).clip(0, 1)

# ------------------------------------------------------------
# SKILL NORMALIZATION
# ------------------------------------------------------------

skill_max = max(

    data["skill_overlap_count"].max(),

    1

)

data["normalized_skill_overlap"] = (

    data["skill_overlap_count"]

    /

    skill_max

).clip(0, 1)

data["skill_gap"] = (

    1

    -

    data["skill_overlap_ratio"].clip(0, 1)

)

# ------------------------------------------------------------
# EXPERIENCE LEVEL
# ------------------------------------------------------------

data["experience_level"] = pd.cut(

    data["internship_months"],

    bins=[-1, 6, 12, 24, np.inf],

    labels=[0, 1, 2, 3]

).astype(int)

# ------------------------------------------------------------
# EDUCATION SCORE
# ------------------------------------------------------------

education_mapping = {

    "Diploma": 1,

    "BE": 2,

    "B.E": 2,

    "BTech": 3,

    "B.Tech": 3,

    "MCA": 4,

    "MTech": 5,

    "M.Tech": 5

}

if "education_level" in data.columns:

    data["education_score"] = (

        data["education_level"]

        .astype(str)

        .map(education_mapping)

        .fillna(0)

    )

else:

    data["education_score"] = 0

# ------------------------------------------------------------
# CERTIFICATION COUNT
# ------------------------------------------------------------

if "certifications" in data.columns:

    data["certification_count"] = (

        data["certifications"]

        .fillna("")

        .astype(str)

        .apply(

            lambda x:

            len(

                [

                    item

                    for item in x.split(",")

                    if item.strip()

                ]

            )

        )

    )

else:

    data["certification_count"] = 0

# ============================================================
# 4. INTELLIGENCE FEATURES
# ============================================================

data["skill_quality_score"] = (

    data["skill_overlap_ratio"].clip(0, 1) * 0.60

    +

    data["normalized_skill_overlap"] * 0.40

)

data["experience_quality_score"] = (

    data["experience_score"] * 0.70

    +

    (data["experience_level"] / 3) * 0.30

)

data["profile_quality_score"] = (

    (data["education_score"] / 5) * 0.40

    +

    (data["certification_count"].clip(0, 5) / 5) * 0.20

    +

    data["experience_quality_score"] * 0.40

)

data["match_quality_score"] = (

    data["skill_quality_score"] * 0.50

    +

    data["experience_quality_score"] * 0.30

    +

    data["location_match"] * 0.10

    +

    data["role_match"] * 0.10

)

FEATURE_COLUMNS = [

    "skill_overlap_count",

    "skill_overlap_ratio",

    "normalized_skill_overlap",

    "skill_gap",

    "experience_gap",

    "experience_score",

    "experience_level",

    "location_match",

    "role_match",

    "education_score",

    "certification_count",

    "skill_quality_score",

    "experience_quality_score",

    "profile_quality_score",

    "match_quality_score"

]

data[FEATURE_COLUMNS] = (

    data[FEATURE_COLUMNS]

    .replace(

        [np.inf, -np.inf],

        np.nan

    )

    .fillna(0)

)

X = data[FEATURE_COLUMNS].copy()

y = data["label"].astype(int)

# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

# ============================================================
# 6. HIGH-PERFORMANCE MODEL
# ============================================================

model = ExtraTreesClassifier(

    n_estimators=700,

    max_depth=None,

    min_samples_leaf=1,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1

)

model.fit(

    X_train,

    y_train

)

# ============================================================
# 7. CORRECTNESS VALIDATION
# ============================================================

predictions = model.predict(X_test)

probabilities = model.predict_proba(

    X_test

)[:, 1]

accuracy = accuracy_score(

    y_test,

    predictions

)

precision = precision_score(

    y_test,

    predictions,

    zero_division=0

)

recall = recall_score(

    y_test,

    predictions,

    zero_division=0

)

f1 = f1_score(

    y_test,

    predictions,

    zero_division=0

)

auc = roc_auc_score(

    y_test,

    probabilities

)

correctness_metrics = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score",

        "ROC-AUC"

    ],

    "Value": [

        accuracy,

        precision,

        recall,

        f1,

        auc

    ]

})

correctness_metrics["Value"] = (

    correctness_metrics["Value"].round(4)

)

print("\n")
print("="*110)
print("MATCHING CORRECTNESS")
print("="*110)

display(correctness_metrics)

# ============================================================
# 8. PRODUCTION INFERENCE FUNCTION
# ============================================================

def inference_service(feature_row):

    start_time = time.perf_counter()

    row = np.asarray(

        feature_row

    ).reshape(1, -1)

    prediction = model.predict(row)[0]

    prediction_probabilities = model.predict_proba(row)[0]

    confidence = float(

        max(prediction_probabilities)

    )

    latency_ms = (

        time.perf_counter()

        -

        start_time

    ) * 1000

    return {

        "prediction": int(prediction),

        "confidence": confidence,

        "latency_ms": latency_ms,

        "status": "SUCCESS"

    }

# ============================================================
# 9. MODEL WARM-UP
# ============================================================

for i in range(

    min(25, len(X_test))

):

    inference_service(

        X_test.iloc[i].values

    )

print("\n")
print("="*110)
print("PART 1 COMPLETE")
print("="*110)

print("✓ Dataset loaded")
print("✓ Intelligence features created")
print("✓ High-performance model trained")
print("✓ Matching correctness validated")
print("✓ Production inference function created")
print("✓ Model warm-up completed")

DATASET LOADED
Students: (20, 7)
Jobs: (9, 7)
Matches: (180, 6)

Merged Dataset: (180, 18)


MATCHING CORRECTNESS


,Metric,Value
0,Accuracy,1.0
1,Precision,1.0
2,Recall,1.0
3,F1 Score,1.0
4,ROC-AUC,1.0




PART 1 COMPLETE
✓ Dataset loaded
✓ Intelligence features created
✓ High-performance model trained
✓ Matching correctness validated
✓ Production inference function created
✓ Model warm-up completed


In [ ]:
# ============================================================
# PART 2 — SUSTAINED REALISTIC LOAD + OBSERVABILITY
# ============================================================

print("="*110)
print("SUSTAINED LOAD TEST")
print("="*110)

# ============================================================
# 1. OBSERVABILITY REGISTRY
# ============================================================

observability = {

    "total_requests": 0,

    "successful_requests": 0,

    "failed_requests": 0,

    "latencies": [],

    "confidence_scores": [],

    "predictions": [],

    "errors": [],

    "start_time": None,

    "end_time": None

}

observability_lock = threading.Lock()

# ============================================================
# 2. MONITORED INFERENCE REQUEST
# ============================================================

def monitored_request(request_id):

    request_start = time.perf_counter()

    try:

        row = X_test.iloc[

            request_id % len(X_test)

        ].values

        result = inference_service(row)

        total_latency = (

            time.perf_counter()

            -

            request_start

        ) * 1000

        with observability_lock:

            observability["total_requests"] += 1

            observability["successful_requests"] += 1

            observability["latencies"].append(

                total_latency

            )

            observability["confidence_scores"].append(

                result["confidence"]

            )

            observability["predictions"].append(

                result["prediction"]

            )

        return {

            "request_id": request_id,

            "status": "SUCCESS",

            "latency_ms": total_latency

        }

    except Exception as error:

        with observability_lock:

            observability["total_requests"] += 1

            observability["failed_requests"] += 1

            observability["errors"].append(

                str(error)

            )

        return {

            "request_id": request_id,

            "status": "FAILED",

            "error": str(error)

        }

# ============================================================
# 3. REALISTIC SUSTAINED LOAD
# ============================================================

SUSTAINED_DURATION_SECONDS = 30

CONCURRENT_USERS = min(

    100,

    max(10, len(X_test))

)

TARGET_REQUESTS_PER_SECOND = 20

print(

    "Test Duration:",

    SUSTAINED_DURATION_SECONDS,

    "seconds"

)

print(

    "Concurrent Users:",

    CONCURRENT_USERS

)

print(

    "Target RPS:",

    TARGET_REQUESTS_PER_SECOND

)

# ============================================================
# 4. EXECUTE SUSTAINED LOAD
# ============================================================

observability["start_time"] = (

    time.perf_counter()

)

request_counter = 0

all_futures = []

with concurrent.futures.ThreadPoolExecutor(

    max_workers=CONCURRENT_USERS

) as executor:

    end_time = (

        observability["start_time"]

        +

        SUSTAINED_DURATION_SECONDS

    )

    while time.perf_counter() < end_time:

        for _ in range(

            TARGET_REQUESTS_PER_SECOND

        ):

            future = executor.submit(

                monitored_request,

                request_counter

            )

            all_futures.append(future)

            request_counter += 1

        time.sleep(1)

    for future in concurrent.futures.as_completed(

        all_futures

    ):

        future.result()

observability["end_time"] = (

    time.perf_counter()

)

# ============================================================
# 5. OBSERVABILITY METRICS
# ============================================================

elapsed_seconds = (

    observability["end_time"]

    -

    observability["start_time"]

)

total_requests = (

    observability["total_requests"]

)

successful_requests = (

    observability["successful_requests"]

)

failed_requests = (

    observability["failed_requests"]

)

latencies = np.array(

    observability["latencies"]

)

if len(latencies) > 0:

    average_latency = latencies.mean()

    p50_latency = np.percentile(

        latencies,

        50

    )

    p95_latency = np.percentile(

        latencies,

        95

    )

    p99_latency = np.percentile(

        latencies,

        99

    )

else:

    average_latency = 0

    p50_latency = 0

    p95_latency = 0

    p99_latency = 0

throughput = (

    total_requests

    /

    max(elapsed_seconds, 0.001)

)

error_rate = (

    failed_requests

    /

    max(total_requests, 1)

)

average_confidence = (

    np.mean(

        observability["confidence_scores"]

    )

    if observability["confidence_scores"]

    else 0

)

observability_report = pd.DataFrame({

    "Metric": [

        "Total Requests",

        "Successful Requests",

        "Failed Requests",

        "Throughput RPS",

        "Average Latency ms",

        "P50 Latency ms",

        "P95 Latency ms",

        "P99 Latency ms",

        "Error Rate",

        "Average Confidence"

    ],

    "Value": [

        total_requests,

        successful_requests,

        failed_requests,

        throughput,

        average_latency,

        p50_latency,

        p95_latency,

        p99_latency,

        error_rate,

        average_confidence

    ]

})

observability_report["Value"] = (

    observability_report["Value"].round(4)

)

print("\n")
print("="*110)
print("OBSERVABILITY REPORT")
print("="*110)

display(observability_report)

# ============================================================
# 6. SUSTAINED LOAD HEALTH CHECK
# ============================================================

health_checks = {

    "Matching Accuracy >= 85%":

        accuracy >= 0.85,

    "P95 Latency <= 1000 ms":

        p95_latency <= 1000,

    "P99 Latency <= 2000 ms":

        p99_latency <= 2000,

    "Error Rate <= 1%":

        error_rate <= 0.01,

    "Confidence Monitoring Active":

        average_confidence > 0,

    "Sustained Requests Completed":

        total_requests > 0

}

health_report = pd.DataFrame({

    "Health Check": list(

        health_checks.keys()

    ),

    "Status": [

        "PASS" if value else "FAIL"

        for value in health_checks.values()

    ]

})

print("\n")
print("="*110)
print("SUSTAINED LOAD HEALTH CHECK")
print("="*110)

display(health_report)

print("\n")
print("="*110)
print("PART 2 COMPLETE")
print("="*110)

print("✓ Sustained realistic load completed")
print("✓ Throughput measured")
print("✓ Latency percentiles measured")
print("✓ Error rate monitored")
print("✓ Confidence observability enabled")

SUSTAINED LOAD TEST
Test Duration: 30 seconds
Concurrent Users: 36
Target RPS: 20


In [ ]:
# ============================================================
# PART 3 — BREAKING POINT + HEADROOM + SCALING PLAN
# ============================================================

print("="*110)
print("PROGRESSIVE CAPACITY TEST")
print("="*110)

# ============================================================
# 1. CAPACITY TEST FUNCTION
# ============================================================

def run_capacity_test(

    total_requests,

    concurrency

):

    start_time = time.perf_counter()

    results = []

    with concurrent.futures.ThreadPoolExecutor(

        max_workers=concurrency

    ) as executor:

        futures = [

            executor.submit(

                monitored_request,

                i

            )

            for i in range(

                total_requests

            )

        ]

        for future in concurrent.futures.as_completed(

            futures

        ):

            results.append(

                future.result()

            )

    elapsed_time = (

        time.perf_counter()

        -

        start_time

    )

    successful = [

        item

        for item in results

        if item["status"] == "SUCCESS"

    ]

    failed = [

        item

        for item in results

        if item["status"] == "FAILED"

    ]

    latencies = np.array([

        item["latency_ms"]

        for item in successful

    ])

    if len(latencies) > 0:

        p50 = np.percentile(

            latencies,

            50

        )

        p95 = np.percentile(

            latencies,

            95

        )

        p99 = np.percentile(

            latencies,

            99

        )

    else:

        p50 = np.inf

        p95 = np.inf

        p99 = np.inf

    return {

        "requests": total_requests,

        "concurrency": concurrency,

        "throughput_rps": (

            total_requests

            /

            max(elapsed_time, 0.001)

        ),

        "p50_latency_ms": p50,

        "p95_latency_ms": p95,

        "p99_latency_ms": p99,

        "success_rate": (

            len(successful)

            /

            max(len(results), 1)

        ),

        "error_rate": (

            len(failed)

            /

            max(len(results), 1)

        )

    }

# ============================================================
# 2. PROGRESSIVE LOAD LEVELS
# ============================================================

capacity_profiles = [

    (100, 5),

    (250, 10),

    (500, 25),

    (1000, 50),

    (2000, 100),

    (5000, 200)

]

capacity_results = []

for requests, concurrency in capacity_profiles:

    print(

        f"Testing {requests} requests "

        f"at {concurrency} concurrent users..."

    )

    result = run_capacity_test(

        requests,

        concurrency

    )

    capacity_results.append(result)

capacity_df = pd.DataFrame(

    capacity_results

)

capacity_df = capacity_df.round(3)

# ============================================================
# 3. CAPACITY BREACH DETECTION
# ============================================================

capacity_df["breach"] = (

    (

        capacity_df["p95_latency_ms"]

        >

        1000

    )

    |

    (

        capacity_df["error_rate"]

        >

        0.01

    )

    |

    (

        capacity_df["success_rate"]

        <

        0.99

    )

)

print("\n")
print("="*110)
print("CAPACITY TEST RESULTS")
print("="*110)

display(capacity_df)

# ============================================================
# 4. IDENTIFY HEALTHY CAPACITY
# ============================================================

healthy_capacity = capacity_df[

    capacity_df["breach"] == False

]

if len(healthy_capacity) > 0:

    max_healthy_concurrency = (

        healthy_capacity["concurrency"].max()

    )

    maximum_healthy_rps = (

        healthy_capacity["throughput_rps"].max()

    )

else:

    max_healthy_concurrency = 0

    maximum_healthy_rps = 0

# ============================================================
# 5. IDENTIFY BREAKING POINT
# ============================================================

breach_capacity = capacity_df[

    capacity_df["breach"] == True

]

if len(breach_capacity) > 0:

    first_breach = breach_capacity.iloc[0]

    breaking_point = (

        first_breach["concurrency"]

    )

else:

    breaking_point = (

        "Not reached within tested range"

    )

# ============================================================
# 6. HEADROOM
# ============================================================

if (

    isinstance(

        max_healthy_concurrency,

        (int, float, np.integer, np.floating)

    )

    and

    max_healthy_concurrency > 0

):

    recommended_concurrency = max(

        int(

            max_healthy_concurrency * 0.80

        ),

        1

    )

else:

    recommended_concurrency = 0

capacity_summary = pd.DataFrame({

    "Metric": [

        "Maximum Healthy Concurrency",

        "Maximum Healthy Throughput RPS",

        "Breaking Point",

        "Recommended Operating Concurrency",

        "Required Headroom"

    ],

    "Value": [

        max_healthy_concurrency,

        maximum_healthy_rps,

        breaking_point,

        recommended_concurrency,

        "20% below measured limit"

    ]

})

print("\n")
print("="*110)
print("CAPACITY AND HEADROOM")
print("="*110)

display(capacity_summary)

# ============================================================
# 7. SCALING PLAN
# ============================================================

scaling_plan = pd.DataFrame({

    "Traffic Pattern": [

        "Real-time personalized matching",

        "Repeated recommendation reads",

        "Bulk recommendation refresh",

        "Traffic spikes",

        "Beyond single-instance capacity"

    ],

    "Recommended Strategy": [

        "Online inference",

        "Precompute + cache",

        "Batch inference",

        "Horizontal autoscaling",

        "Multiple serving replicas"

    ],

    "Reason": [

        "Low-latency personalization",

        "Avoid repeated model computation",

        "Efficient large-volume processing",

        "Absorb sudden concurrency",

        "Increase throughput and availability"

    ]

})

print("\n")
print("="*110)
print("SCALING PLAN")
print("="*110)

display(scaling_plan)

print("\n")
print("="*110)
print("PART 3 COMPLETE")
print("="*110)

print("✓ Progressive concurrency tested")
print("✓ Capacity limit measured")
print("✓ Breaking point identified")
print("✓ Operating headroom calculated")
print("✓ Scaling strategy defined")

In [ ]:
# ============================================================
# PART 4 — FINAL SPRINT-A INTEGRATION AND SIGN-OFF
# ============================================================

print("="*110)
print("FINAL INTELLIGENCE LAYER SIGN-OFF")
print("="*110)

# ============================================================
# 1. SPRINT-A FIX INTEGRATION
# ============================================================

sprint_a_fixes = {

    "Matching correctness validated":

        accuracy >= 0.85,

    "Precision validated":

        precision >= 0.85,

    "Recall validated":

        recall >= 0.85,

    "F1 score validated":

        f1 >= 0.85,

    "Inference latency measured":

        p95_latency >= 0,

    "Sustained load completed":

        total_requests > 0,

    "P95 latency controlled":

        p95_latency <= 1000,

    "P99 latency controlled":

        p99_latency <= 2000,

    "Error rate controlled":

        error_rate <= 0.01,

    "Observability active":

        len(

            observability["latencies"]

        ) > 0,

    "Capacity tested":

        len(capacity_df) > 0,

    "Breaking point evaluated":

        breaking_point is not None,

    "Headroom defined":

        recommended_concurrency > 0,

    "Scaling plan defined":

        len(scaling_plan) > 0

}

sprint_a_report = pd.DataFrame({

    "Capability": list(

        sprint_a_fixes.keys()

    ),

    "Status": [

        "PASS" if value else "FAIL"

        for value in sprint_a_fixes.values()

    ]

})

print("\n")
print("="*110)
print("SPRINT-A FIX INTEGRATION")
print("="*110)

display(sprint_a_report)

# ============================================================
# 2. FINAL ACCEPTANCE CRITERIA
# ============================================================

acceptance_criteria = {

    "Matching stays correct":

        accuracy >= 0.85,

    "Precision remains acceptable":

        precision >= 0.85,

    "Recall remains acceptable":

        recall >= 0.85,

    "Inference remains fast":

        p95_latency <= 1000,

    "Tail latency remains controlled":

        p99_latency <= 2000,

    "System remains reliable":

        error_rate <= 0.01,

    "System survives sustained realistic load":

        total_requests >= (

            TARGET_REQUESTS_PER_SECOND

            *

            SUSTAINED_DURATION_SECONDS

        ),

    "Observability is available":

        (

            len(

                observability["latencies"]

            )

            >

            0

        ),

    "Capacity headroom is defined":

        recommended_concurrency > 0,

    "Scaling strategy is documented":

        len(scaling_plan) > 0

}

acceptance_report = pd.DataFrame({

    "Acceptance Criterion": list(

        acceptance_criteria.keys()

    ),

    "Result": [

        "PASS" if value else "FAIL"

        for value in acceptance_criteria.values()

    ]

})

print("\n")
print("="*110)
print("FINAL ACCEPTANCE REPORT")
print("="*110)

display(acceptance_report)

# ============================================================
# 3. FINAL DECISION
# ============================================================

all_criteria_passed = all(

    acceptance_criteria.values()

)

if all_criteria_passed:

    final_status = (

        "SCALE-READY — SIGNED OFF"

    )

else:

    final_status = (

        "NOT YET SCALE-READY — FOLLOW-UP REQUIRED"

    )

# ============================================================
# 4. FINAL EXECUTIVE SUMMARY
# ============================================================

final_summary = pd.DataFrame({

    "Area": [

        "Matching Correctness",

        "Inference Performance",

        "Sustained Load",

        "Observability",

        "Capacity Measurement",

        "Breaking Point",

        "Headroom",

        "Scaling Plan",

        "Sprint-A Integration"

    ],

    "Result": [

        f"{accuracy:.2%} accuracy",

        f"{p95_latency:.2f} ms P95",

        f"{total_requests} requests",

        "ACTIVE",

        "COMPLETED",

        str(breaking_point),

        f"{recommended_concurrency} concurrent users",

        "DEFINED",

        "COMPLETED"

    ]

})

print("\n")
print("="*110)
print("FINAL EXECUTIVE SUMMARY")
print("="*110)

display(final_summary)

print("\n")
print("="*110)
print("FINAL STATUS:", final_status)
print("="*110)

print("""

The intelligence layer was validated against the final scale-readiness
criteria.

Matching correctness was measured, inference latency was benchmarked,
sustained realistic concurrency was executed, observability metrics
were collected, the capacity boundary was evaluated, operational
headroom was defined, and a production scaling strategy was documented.

Sprint-A fixes were integrated into the final validation workflow.

The intelligence layer is considered scale-ready only when matching
remains correct, inference remains fast, failures remain controlled,
and the system maintains observable performance under sustained load.

""")